# ST 554 Project 2
*By: Cass Crews*

In this notebook, we will explore the capabilities of the `PySpark` API. More specifically, we will demonstrate the functionality of the `PySpark.sql` and `PySpark.pandas` modules, two key modules that bridge the gap between Python's functionalities and the big data capabilities of Apache Spark. 

In Part I, we will use a custom class, specifically built for this notebook and stored in the same repository, to conduct basic data validation on the SQL-style dataframes central to `PySpark.sql`. 

In Part II, we will complete the same data manipulation process using the `PySpark.sql` and `PySpark.pandas` modules, highlighting their similarities and differences. 

Before we get started, we need to read in the modules and sub-modules we'll use throughout the notebook. We also need to initialize a Spark session. 

In [2]:
# Importing key modules and sub-modules, and initializing a Spark session
import numpy as np
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from functools import reduce
from pyspark.sql.types import *
import pandas as pd
from pyspark.sql import SparkSession
import pyspark.pandas as ps
ps.set_option('compute.fail_on_ansi_mode', False)
spark = SparkSession.builder.master('local[*]').appName('my_app') \
.config("spark.sql.ansi.enabled", "false").getOrCreate()

/opt/tljh/user/envs/pySpark3/lib/python3.9/site-packages/pyspark/pandas/__init__.py:43: UserWarning: 'PYARROW_IGNORE_TIMEZONE' environment variable was not set. It is required to set this environment variable to '1' in both driver and executor sides if you use pyarrow>=2.0.0. pandas-on-Spark will set it for you but it does not work if there is a Spark context already launched.
  warnings.warn(
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/23 20:44:35 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/23 20:44:36 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/03/23 20:44:36 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/03/23 20:44:36 WARN Utils: Service 'SparkUI' cou

## Part I

While `PySpark.sql` is incredibly useful for accessing and manipulating databases, some steps of a standard exploratory data analysis (EDA) can be more tedious than they are when working in R's tidyverse or when utilizing the `pandas` module. Thus, Part I explores a custom class, `SparkDataCheck`, created specifically for this notebook to highlight the abilities to use classes and associated methods to simplify common EDA steps. The source script for `SparkDataCheck` can be found [here](https://github.com/jccrews256/ST-554-Project-2/blob/main/SparkDataCheck.py). 

`SparkDataCheck` is designed to be initialized in three convenient ways, with each resulting in a `.df` attribute that stores a dataset in a `PySpark.sql` dataframe:

- `.__init__()`: This initializing method directly takes in a `PySpark.sql` dataframe
- `.from_csv()`: This class method creates an instance of our custom class while reading in the data from a csv file
    - In addition to the file path, we need to provide the Spark session information
- `.from_pddf()`: This class method creates an instance of our custom class from a `pandas` dataframe
    - In addition to the dataframe name, we need to provide the Spark session information

Of course, to utilize any of these three methods, we need to make the class accessible in our environment. Let's do that now. 

*Note: The `air.csv` and `SparkDataCheck.py` files will need to be available in the reader's environment to run all code chunks in Part I.*

In [3]:
from SparkDataCheck import *

Before we begin exploring objects of the `SparkDataCheck` class, let's print the docstring to fully understand what the class offers. 

In [4]:
# Printing class docstring
print(SparkDataCheck.__doc__)


    A class designed to assist the user in validating a PySpark.sql dataframe,
    stored as the attribute df. 
    
    In addition to using __init__ to directly initialize the class with a 
    pre-existing PySpark.sql dataframe, there are two convenient class methods
    to create an object of class SparkDataCheck:
        - from_csv: a class method that takes in the Spark session (spark) and
        a file path to a csv data file (csv_path) as inputs to create the object,
        storing the dataset as a PySpark.sql dataframe in df
        - from_pddf: a class method that takes in the Spark session (spark) and
        a pandas dataframe (pd_df) as inputs to create the object, storing the 
        dataset as a PySpark.sql dataframe in df
        
    Note that these two class methods will not produce identical results because
    pandas and PySpark.sql exhibit different csv-to-dataframe behaviors. 
    
    Methods:
        - in_range: a method that takes in a numeric column (colum

The docstring notes the three ways of initializing the class, as well as five methods. After we create a `SparkDataCheck` object, we will thoroughly explore each method. 

#### Initializing Class Using `.from_csv()`

Let's utilize the `.from_csv()` class method to initialize our custom class while reading in a csv file of air pollution readings. This dataset was developed to test the efficacy of using low-cost chemical sensors to estimate true pollutant concentrations in situations where pollution monitoring systems are cost-prohibitive. Thus, the data contains hourly true concentrations and sensor readings for multiple pollutants in a single location. We will not explore the context further as our focus is on exploring the custom class. However, interested readers can access the data [here](https://archive.ics.uci.edu/dataset/360/air+quality) and learn more about the study [here](https://www.sciencedirect.com/science/article/abs/pii/S0925400507007691). 

In [5]:
# Creating a SparkDataCheck object containing the air quality data
test_data_sql = SparkDataCheck.from_csv(spark, "air.csv")

Let's print the first 25 rows of the dataset to gain a better understanding of the data structure. 

In [6]:
# Extracting the first 25 rows of the data
test_data_sql.df.show(25)

+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|_c0|     Date|               Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|  0|3/10/2004|2026-03-23 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|
|  1|3/10/2004|2026-03-23 19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1559|        972|13.3|47.7|0.7255|
|  2|3/10/2004|2026-03-23 20:00:00|   2.2|       1402|      88|     9.0|          939|    131|        1140|    114|        1555|       1074|11.9|54.0|0.7502|
|  3|3/10/2004|2026-03-23 21:00:00|   2.2|       137

26/03/23 20:44:40 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-jccrews@ncsu.edu/air.csv


Note the warning, which indicates the first column in the file does not have a name. This is simply the row index, so the warning is not concerning. In the next code chunk, we'll adjust the settings of our Spark session so that warnings are no longer printed.

Another interesting note is that the `Time` values are assigned the date corresponding to when the code is run. This is because `PySpark` coerces times to time stamps and uses the date of run as a placeholder date.

The rows printed above indicate an idiosyncrasy of the dataset that we will need to handle before we explore the methods of our custom class: missing values are coded -200.  Let's loop over the atmospheric measurements (`CO(GT)` through `AH`) to convert each `-200` to `NULL`. In the process, we will update the variable names because the "." in some of the names will be problematic for `PySpark`. 

In [7]:
# Suppressing future warnings but not errors
spark.sparkContext.setLogLevel("ERROR")

# Specifying clean variable names
clean_names = ['Index', 'Date', 'Time', 'CO_true', 'CO_sensor', 'NMHC_true', 'C6H6_true', 
               'NMHC_sensor', 'NOx_true', 'NOx_sensor', 'NO2_true', 'NO2_sensor', 'O3_sensor', 
               'Temp', 'Rel_Humid', 'Abs_Humid']

# Updating names and converting the missing values (labeled -200) to NULL
for i in range(len(test_data_sql.df.columns)):
    # Updating names
    test_data_sql.df = test_data_sql.df.withColumnRenamed(test_data_sql.df.columns[i], clean_names[i])
    
    # Converting missing values for the measurements
    if i >= 3:
        test_data_sql.df = test_data_sql.df.withColumn(clean_names[i], F.when(test_data_sql.df[clean_names[i]] == -200, None).otherwise(test_data_sql.df[clean_names[i]]))        

Let's print out the first 25 rows again to ensure our updates worked. 

In [8]:
# Extracting the first 25 rows of the data
test_data_sql.df.show(25)

+-----+---------+-------------------+-------+---------+---------+---------+-----------+--------+----------+--------+----------+---------+----+---------+---------+
|Index|     Date|               Time|CO_true|CO_sensor|NMHC_true|C6H6_true|NMHC_sensor|NOx_true|NOx_sensor|NO2_true|NO2_sensor|O3_sensor|Temp|Rel_Humid|Abs_Humid|
+-----+---------+-------------------+-------+---------+---------+---------+-----------+--------+----------+--------+----------+---------+----+---------+---------+
|    0|3/10/2004|2026-03-23 18:00:00|    2.6|     1360|      150|     11.9|       1046|     166|      1056|     113|      1692|     1268|13.6|     48.9|   0.7578|
|    1|3/10/2004|2026-03-23 19:00:00|    2.0|     1292|      112|      9.4|        955|     103|      1174|      92|      1559|      972|13.3|     47.7|   0.7255|
|    2|3/10/2004|2026-03-23 20:00:00|    2.2|     1402|       88|      9.0|        939|     131|      1140|     114|      1555|     1074|11.9|     54.0|   0.7502|
|    3|3/10/2004|2026-

We've now made all the updates we need to explore `SparkDataCheck`'s methods. We'll be doing so with variables names that are easier to interpret for readers who are familiar with chemical symbols. For example, the `CO_true` column contains the true carbon monoxide concentrations. 

#### Testing the `.in_range()` Method

Let's start with the `.in_range()` method, which adds a new column to the dataframe indicating whether the values of a numeric column in the dataset (`column`) are between the `lower` and `upper` inputs, inclusive. At least one of these bounding inputs must be provided. 

For our first use of `.in_range()`, let's identify which true carbon monoxide concentrations are between 1 and 3 milligrams per cubic meter. 

In [9]:
# Adding new boolean column indicating whether CO_true between 1 and 3
test_data_sql.in_range(column = "CO_true", lower = 1, upper = 3).df.show(25)

+-----+---------+-------------------+-------+---------+---------+---------+-----------+--------+----------+--------+----------+---------+----+---------+---------+--------------+
|Index|     Date|               Time|CO_true|CO_sensor|NMHC_true|C6H6_true|NMHC_sensor|NOx_true|NOx_sensor|NO2_true|NO2_sensor|O3_sensor|Temp|Rel_Humid|Abs_Humid|CO_true_in_1_3|
+-----+---------+-------------------+-------+---------+---------+---------+-----------+--------+----------+--------+----------+---------+----+---------+---------+--------------+
|    0|3/10/2004|2026-03-23 18:00:00|    2.6|     1360|      150|     11.9|       1046|     166|      1056|     113|      1692|     1268|13.6|     48.9|   0.7578|          true|
|    1|3/10/2004|2026-03-23 19:00:00|    2.0|     1292|      112|      9.4|        955|     103|      1174|      92|      1559|      972|13.3|     47.7|   0.7255|          true|
|    2|3/10/2004|2026-03-23 20:00:00|    2.2|     1402|       88|      9.0|        939|     131|      1140|   

When we look at the 25 observations we are becoming rather familiar with, we now see a column of "true" and "false" values. I must say, this new column has a great name! 

One thing we should note: when the input column is `NULL`, the new column is also `NULL`. This will be convenient if we want to count true and false values without worrying about including missing values. 

Next, let's generate a new column that indicates when `CO_true` is greater than or equal to 1, with no upper bound. 

In [10]:
# Adding new boolean column indicating whether CO_true above 1
test_data_sql.in_range(column = "CO_true", lower = 1).df.show(25)

+-----+---------+-------------------+-------+---------+---------+---------+-----------+--------+----------+--------+----------+---------+----+---------+---------+--------------+---------------+
|Index|     Date|               Time|CO_true|CO_sensor|NMHC_true|C6H6_true|NMHC_sensor|NOx_true|NOx_sensor|NO2_true|NO2_sensor|O3_sensor|Temp|Rel_Humid|Abs_Humid|CO_true_in_1_3|CO_true_above_1|
+-----+---------+-------------------+-------+---------+---------+---------+-----------+--------+----------+--------+----------+---------+----+---------+---------+--------------+---------------+
|    0|3/10/2004|2026-03-23 18:00:00|    2.6|     1360|      150|     11.9|       1046|     166|      1056|     113|      1692|     1268|13.6|     48.9|   0.7578|          true|           true|
|    1|3/10/2004|2026-03-23 19:00:00|    2.0|     1292|      112|      9.4|        955|     103|      1174|      92|      1559|      972|13.3|     47.7|   0.7255|          true|           true|
|    2|3/10/2004|2026-03-23 20

Comparing the two Boolean columns we have created so far, it seems a lower bound of 1 unit is more restrictive than an upper bound of 3 -- at least for the first 25 observations. 

Let's create another column that indicates whether the concentrations are below 3 units, isolating these seemingly rare occurrences above 3; we'll subset the columns we `show` so that the printed table is not too crowded. I suspect the reader is wondering how these columns will be used. Do not fret, as another method for this class can be used to count "true" and "false" values. 

In [11]:
# Adding new boolean column indicating whether CO_true below 3
test_data_sql.in_range(column = "CO_true", upper = 3).df.select("Date", "Time", "CO_true", "CO_true_in_1_3", "CO_true_above_1", "CO_true_below_3").show(25)

+---------+-------------------+-------+--------------+---------------+---------------+
|     Date|               Time|CO_true|CO_true_in_1_3|CO_true_above_1|CO_true_below_3|
+---------+-------------------+-------+--------------+---------------+---------------+
|3/10/2004|2026-03-23 18:00:00|    2.6|          true|           true|           true|
|3/10/2004|2026-03-23 19:00:00|    2.0|          true|           true|           true|
|3/10/2004|2026-03-23 20:00:00|    2.2|          true|           true|           true|
|3/10/2004|2026-03-23 21:00:00|    2.2|          true|           true|           true|
|3/10/2004|2026-03-23 22:00:00|    1.6|          true|           true|           true|
|3/10/2004|2026-03-23 23:00:00|    1.2|          true|           true|           true|
|3/11/2004|2026-03-23 00:00:00|    1.2|          true|           true|           true|
|3/11/2004|2026-03-23 01:00:00|    1.0|          true|           true|           true|
|3/11/2004|2026-03-23 02:00:00|    0.9|    

We now have three boolean columns effectively splitting the empirical distribution of true carbon monoxide concentration into fixed regions. We will soon use these columns to understand the likelihoods of falling into each region.  

Before moving onto the next method, let's see what happens when we pass a string column to `.in_range`. 

In [12]:
# Testing case when input column is not numeric
test_data_sql.in_range(column = "Date", lower = 1).df.drop("Index").show(25)

Column must be a numeric type
+---------+-------------------+-------+---------+---------+---------+-----------+--------+----------+--------+----------+---------+----+---------+---------+--------------+---------------+---------------+
|     Date|               Time|CO_true|CO_sensor|NMHC_true|C6H6_true|NMHC_sensor|NOx_true|NOx_sensor|NO2_true|NO2_sensor|O3_sensor|Temp|Rel_Humid|Abs_Humid|CO_true_in_1_3|CO_true_above_1|CO_true_below_3|
+---------+-------------------+-------+---------+---------+---------+-----------+--------+----------+--------+----------+---------+----+---------+---------+--------------+---------------+---------------+
|3/10/2004|2026-03-23 18:00:00|    2.6|     1360|      150|     11.9|       1046|     166|      1056|     113|      1692|     1268|13.6|     48.9|   0.7578|          true|           true|           true|
|3/10/2004|2026-03-23 19:00:00|    2.0|     1292|      112|      9.4|        955|     103|      1174|      92|      1559|      972|13.3|     47.7|   0.725

This is useful functionality: the method returns a message indicating the column must be a numeric type, but it also returns the unmodified SparkDataCheck object so that we can proceed in chaining operations with no issues.

#### Testing the `.in_set()` Method

In many ways, the `.in_set()` method is the categorical variable analog to `.in_range()`. It adds a Boolean column to `df` indicating whether the value of a `string` column (`column`) is within a set of `levels`. To capture the context of the new column, the user can optionally provide a `tag` that will be incorporated into the column name. 

Let's test the `.in_set()` method by indicating when the date is "3/11/2004"; we'll set `tag` to "mar_11" for future reference. As our dataframe is becoming rather wide, we will again subset the columns printed. 

In [13]:
# Adding new boolean column indicating when Date is 3/11/2004
test_data_sql.in_set(column = "Date", levels = "3/11/2004", tag = "mar_11").df \
    .select("Date", "Time", "Date_in_set_mar_11").show(25)

+---------+-------------------+------------------+
|     Date|               Time|Date_in_set_mar_11|
+---------+-------------------+------------------+
|3/10/2004|2026-03-23 18:00:00|             false|
|3/10/2004|2026-03-23 19:00:00|             false|
|3/10/2004|2026-03-23 20:00:00|             false|
|3/10/2004|2026-03-23 21:00:00|             false|
|3/10/2004|2026-03-23 22:00:00|             false|
|3/10/2004|2026-03-23 23:00:00|             false|
|3/11/2004|2026-03-23 00:00:00|              true|
|3/11/2004|2026-03-23 01:00:00|              true|
|3/11/2004|2026-03-23 02:00:00|              true|
|3/11/2004|2026-03-23 03:00:00|              true|
|3/11/2004|2026-03-23 04:00:00|              true|
|3/11/2004|2026-03-23 05:00:00|              true|
|3/11/2004|2026-03-23 06:00:00|              true|
|3/11/2004|2026-03-23 07:00:00|              true|
|3/11/2004|2026-03-23 08:00:00|              true|
|3/11/2004|2026-03-23 09:00:00|              true|
|3/11/2004|2026-03-23 10:00:00|

This is another useful method when we want to identify observations with certain characteristics. 

If we want to identify all observations, corresponding to 6am, we can pull the hour from `Time` and use that in conjunction with `.in_set`. 

In [14]:
# Extracting the hour as a new column
test_data_sql.df = test_data_sql.df.withColumn("Hour", F.hour("Time").cast("string"))

# Adding new boolean column indicating when the hour is 6
test_data_sql.in_set(column = "Hour", levels = "6", tag = "6am").df \
    .select("Date", "Time", "Hour_in_set_6am").show(25)

+---------+-------------------+---------------+
|     Date|               Time|Hour_in_set_6am|
+---------+-------------------+---------------+
|3/10/2004|2026-03-23 18:00:00|          false|
|3/10/2004|2026-03-23 19:00:00|          false|
|3/10/2004|2026-03-23 20:00:00|          false|
|3/10/2004|2026-03-23 21:00:00|          false|
|3/10/2004|2026-03-23 22:00:00|          false|
|3/10/2004|2026-03-23 23:00:00|          false|
|3/11/2004|2026-03-23 00:00:00|          false|
|3/11/2004|2026-03-23 01:00:00|          false|
|3/11/2004|2026-03-23 02:00:00|          false|
|3/11/2004|2026-03-23 03:00:00|          false|
|3/11/2004|2026-03-23 04:00:00|          false|
|3/11/2004|2026-03-23 05:00:00|          false|
|3/11/2004|2026-03-23 06:00:00|           true|
|3/11/2004|2026-03-23 07:00:00|          false|
|3/11/2004|2026-03-23 08:00:00|          false|
|3/11/2004|2026-03-23 09:00:00|          false|
|3/11/2004|2026-03-23 10:00:00|          false|
|3/11/2004|2026-03-23 11:00:00|         

We can now easily identify all measurements taken at 6am! It will be interesting to see how pollution at this time compares to pollution in the other 23 hours of the day. 

Let's see what happens when we provide a numeric column. We'll drop a few measurements so the printed table isn't wider than the notebook itself. 

In [15]:
# Testing behavior when column is numeric
test_data_sql.in_set(column = "CO_true", levels = "5", tag = "bad_input").df \
    .drop('Index', 'CO_sensor', 'NMHC_true', 'C6H6_true', 
               'NMHC_sensor', 'NOx_true', 'NOx_sensor', 
          'NO2_true', 'NO2_sensor', 'O3_sensor').show(25)

Column must be a string type
+---------+-------------------+-------+----+---------+---------+--------------+---------------+---------------+------------------+----+---------------+
|     Date|               Time|CO_true|Temp|Rel_Humid|Abs_Humid|CO_true_in_1_3|CO_true_above_1|CO_true_below_3|Date_in_set_mar_11|Hour|Hour_in_set_6am|
+---------+-------------------+-------+----+---------+---------+--------------+---------------+---------------+------------------+----+---------------+
|3/10/2004|2026-03-23 18:00:00|    2.6|13.6|     48.9|   0.7578|          true|           true|           true|             false|  18|          false|
|3/10/2004|2026-03-23 19:00:00|    2.0|13.3|     47.7|   0.7255|          true|           true|           true|             false|  19|          false|
|3/10/2004|2026-03-23 20:00:00|    2.2|11.9|     54.0|   0.7502|          true|           true|           true|             false|  20|          false|
|3/10/2004|2026-03-23 21:00:00|    2.2|11.0|     60.0|   0.

Similar to `.in_range`, `.in_set` takes improper column inputs in stride, simply printing the message "Column must be a string type" and returning the unmodified SparkDataCheck object. 

If we want to identify the `NULL` values in the `CO_true` column, we can try to use `.in_set` to flag "NULL" in a string-cast version of `CO_true_in_1_3`. 

In [16]:
# Casting CO_true_in_1_3 to string
test_data_sql.df = test_data_sql.df.withColumn("CO_true_in_1_3_str", F.col("CO_true_in_1_3").cast("string"))

# Adding new boolean column indicating when this new string is NULL
test_data_sql.in_set(column = "CO_true_in_1_3_str", levels = "NULL", tag = "null").df \
    .select("Date", "Time", "CO_true_in_1_3_str", "CO_true_in_1_3_str_in_set_null").show(25)

+---------+-------------------+------------------+------------------------------+
|     Date|               Time|CO_true_in_1_3_str|CO_true_in_1_3_str_in_set_null|
+---------+-------------------+------------------+------------------------------+
|3/10/2004|2026-03-23 18:00:00|              true|                         false|
|3/10/2004|2026-03-23 19:00:00|              true|                         false|
|3/10/2004|2026-03-23 20:00:00|              true|                         false|
|3/10/2004|2026-03-23 21:00:00|              true|                         false|
|3/10/2004|2026-03-23 22:00:00|              true|                         false|
|3/10/2004|2026-03-23 23:00:00|              true|                         false|
|3/11/2004|2026-03-23 00:00:00|              true|                         false|
|3/11/2004|2026-03-23 01:00:00|              true|                         false|
|3/11/2004|2026-03-23 02:00:00|             false|                         false|
|3/11/2004|2026-

That did not accomplish our goal.... It seems we need another tool in the toolbox. 

#### Testing the `.is_null()` Method

The `.is_null()` method was created for the explicit purpose of flagging `NULL` values in a `column`. Let's use that on the original `CO_true_in_1_3`. 

In [17]:
# Adding a boolean column indicating when CO_true_in_1_3 is NULL
test_data_sql.is_null(column = "CO_true_in_1_3").df.select("Date", "Time", 
                                               "CO_true_in_1_3", 
                                               "CO_true_in_1_3_is_null").show(25)

+---------+-------------------+--------------+----------------------+
|     Date|               Time|CO_true_in_1_3|CO_true_in_1_3_is_null|
+---------+-------------------+--------------+----------------------+
|3/10/2004|2026-03-23 18:00:00|          true|                 false|
|3/10/2004|2026-03-23 19:00:00|          true|                 false|
|3/10/2004|2026-03-23 20:00:00|          true|                 false|
|3/10/2004|2026-03-23 21:00:00|          true|                 false|
|3/10/2004|2026-03-23 22:00:00|          true|                 false|
|3/10/2004|2026-03-23 23:00:00|          true|                 false|
|3/11/2004|2026-03-23 00:00:00|          true|                 false|
|3/11/2004|2026-03-23 01:00:00|          true|                 false|
|3/11/2004|2026-03-23 02:00:00|         false|                 false|
|3/11/2004|2026-03-23 03:00:00|         false|                 false|
|3/11/2004|2026-03-23 04:00:00|          NULL|                  true|
|3/11/2004|2026-03-2

Because `.is_null()` can handle any type of column, we can obtain the same result by passing `CO_true` directly. 

In [18]:
# Adding a column indicating when CO_true is NULL
test_data_sql.is_null(column = "CO_true").df.select("Date", "Time", "CO_true", 
                                        "CO_true_in_1_3", "CO_true_in_1_3_is_null", 
                                        "CO_true_is_null").show(25)

+---------+-------------------+-------+--------------+----------------------+---------------+
|     Date|               Time|CO_true|CO_true_in_1_3|CO_true_in_1_3_is_null|CO_true_is_null|
+---------+-------------------+-------+--------------+----------------------+---------------+
|3/10/2004|2026-03-23 18:00:00|    2.6|          true|                 false|          false|
|3/10/2004|2026-03-23 19:00:00|    2.0|          true|                 false|          false|
|3/10/2004|2026-03-23 20:00:00|    2.2|          true|                 false|          false|
|3/10/2004|2026-03-23 21:00:00|    2.2|          true|                 false|          false|
|3/10/2004|2026-03-23 22:00:00|    1.6|          true|                 false|          false|
|3/10/2004|2026-03-23 23:00:00|    1.2|          true|                 false|          false|
|3/11/2004|2026-03-23 00:00:00|    1.2|          true|                 false|          false|
|3/11/2004|2026-03-23 01:00:00|    1.0|          true|      

These two `NULL`-identifying Boolean columns should be identical, and that seems to be the case based on the first 25 rows. 

#### Testing the `.calc_min_max()` Method

How can we extract information that explicitly validates the data, rather than simply generating Boolean columns? One option is the `.calc_min_max()` method, which returns the min and max of one or all numeric columns either unconditionally or conditional on a `groupby` input. The `groupby` input is a single column that should be categorical in nature to prevent excessively fine grouping, but there are no limitations on the formal column type. 

To start, let's extract the unconditional minimums and maximums for each numeric column, which can be done by not providing any additional information to the method. 

In [19]:
# Extracting the unconditional min and max of all numeric columns
test_data_sql.calc_min_max()

,min(Index),max(Index),min(CO_true),max(CO_true),min(CO_sensor),max(CO_sensor),min(NMHC_true),max(NMHC_true),min(C6H6_true),max(C6H6_true),...,min(NO2_sensor),max(NO2_sensor),min(O3_sensor),max(O3_sensor),min(Temp),max(Temp),min(Rel_Humid),max(Rel_Humid),min(Abs_Humid),max(Abs_Humid)
0,0,9356,0.1,11.9,647,2040,7,1189,0.1,63.7,...,551,2775,221,2523,-1.9,44.6,9.2,88.7,0.1847,2.231


Three notes: First, the results are conveniently returned as a `pandas` dataframe, which is reasonable for summary statistics; `PySpark` dataframes are unnecessary here. Second, `Index` is technically numeric, but its min and max only characterize the number of rows (9356+1=9357). Third, this is an overwhelming amount of information. 

Let's generate the minimums and maximums for each numeric variable across the hours of the day simply to demonstrate the ability to generate conditional endpoints for each empirical numeric distribution in this dataset, then transition to single-column evaluations. `.calc_min_max` automatically sorts the dataframe by the `groupby` column, so let's convert `Hour` to an integer to ensure the sorting is intuitive. 

In [20]:
# Casting Hour to numeric for intuitive sorting
test_data_sql.df = test_data_sql.df.withColumn("Hour_int", test_data_sql.df["Hour"].cast("int"))

# Extracting the conditional min and max of all numeric columns across Hour_int
test_data_sql.calc_min_max(groupby = "Hour_int")

,Hour_int,min(Index),max(Index),min(CO_true),max(CO_true),min(CO_sensor),max(CO_sensor),min(NMHC_true),max(NMHC_true),min(C6H6_true),...,min(O3_sensor),max(O3_sensor),min(Temp),max(Temp),min(Rel_Humid),max(Rel_Humid),min(Abs_Humid),max(Abs_Humid),min(Hour_int),max(Hour_int)
22,0,6,9342,0.1,5.9,683,1712,31,275,1.0,...,322,2411,0.2,29.8,14.0,84.1,0.1910,2.1395,0,0
2,1,7,9343,0.1,5.6,692,1751,27,251,0.6,...,307,2519,0.0,28.4,14.5,83.9,0.1847,2.1059,1,1
21,2,8,9344,0.1,5.5,679,1574,23,173,0.4,...,268,1967,-0.1,28.3,17.0,84.6,0.1975,2.1247,2,2
6,3,9,9345,0.1,5.2,676,1458,19,132,0.2,...,227,1960,-1.1,28.8,19.2,86.6,0.2379,2.1074,3,3
13,4,10,9346,0.1,2.8,681,1515,9,96,0.2,...,221,1847,-1.4,28.9,22.5,86.6,0.2326,2.1010,4,4
8,5,11,9347,0.1,2.9,678,1491,7,93,0.1,...,225,1792,-1.3,27.5,23.1,85.7,0.2347,2.1195,5,5
4,6,12,9348,0.1,3.5,655,1453,16,179,0.1,...,232,1680,-1.2,27.5,24.7,85.5,0.2404,2.1170,6,6
16,7,13,9349,0.1,5.7,649,1689,27,754,0.2,...,261,1869,-1.3,27.6,30.9,85.3,0.2478,2.1008,7,7
14,8,14,9350,0.1,7.3,647,1797,36,1129,0.3,...,274,2176,-1.9,29.0,24.0,84.5,0.2478,2.1719,8,8
11,9,15,9351,0.1,8.1,667,1961,35,798,0.4,...,291,2346,-0.5,31.0,20.0,83.9,0.2388,2.1362,9,9


It's great to know this is possible for datasets with fewer numeric variables, but the table provides too much information in this case. 

Let's generate the hour-dependent minimums and maximums for `CO_true` only by passing this variable to `column`. 

In [21]:
# Extracting the conditional min and max of CO_true across Hour_int
test_data_sql.calc_min_max(column = "CO_true", groupby = "Hour_int")

,Hour_int,min(CO_true),max(CO_true)
22,0,0.1,5.9
2,1,0.1,5.6
21,2,0.1,5.5
6,3,0.1,5.2
13,4,0.1,2.8
8,5,0.1,2.9
4,6,0.1,3.5
16,7,0.1,5.7
14,8,0.1,7.3
11,9,0.1,8.1


These results are fascinating; the min values are similar across the day, but the max values show an interesting pattern. In particular, carbon monoxide levels climb to a local peak as people start their days, then hit a higher peak in the evening. The divergence in trends between min and max values may indicate that a few entire days exhibit low polluting activity.

To explore this further, let's use `.in_set` to isolate a random winter Sunday (February 6, 2005) and compare its min and max carbon monoxide levels to the min and max for all other days. 

In [22]:
# Adding new boolean column indicating when Date is 2/6/2005
test_data_sql.in_set("Date", "2/6/2005", tag = "feb_6")

# Extracting the conditional min and max of CO_true across Hour_int
test_data_sql.calc_min_max(column = "CO_true", groupby = "Date_in_set_feb_6")

,Date_in_set_feb_6,min(CO_true),max(CO_true)
1,False,0.1,11.9
0,True,0.6,2.0


Indeed, carbon monoxide levels are relatively low across February 6th. 

As a final test, let's see what happens if we provide a string column to `column`. 

In [23]:
# Testing a string column input to column
test_data_sql.calc_min_max(column = "Hour")

Column must be a numeric type


This is convenient: the method simply prints a message indicating `column` must be numeric. If it instead threw an error, an attempt to run all code chunks in this notebook would stop at this chunk. 

#### Testing the `.count_combos()` Method

The first three methods we explored create Boolean columns based on some condition. It would be helpful to know the breakdown between true's and false's. `.count_combos` can help us do just that. This method takes in up to two string columns (`column1` and `column2`). If only `column1` is input, the method returns a `pandas` dataframe with counts for each level of the column. If two columns are input, the method returns a dataframe with counts for each combination of the two columns' levels. 

Let's use the method to count the number of observations with `CO_true` values between 1 and 3 units. To do so, we can use the string-cast version of `CO_true_in_1_3` we already created. 

In [24]:
# Counting observations where CO_true between 1 and 3 (and not)
test_data_sql.count_combos(column1 = "CO_true_in_1_3_str")

,CO_true_in_1_3_str,count
0,None,1683
1,false,3271
2,true,4403


It seems the carbon monoxide concentration is more likely to be between 1 and 3 units than not. Also, nearly 1,700 observations are missing a value for `CO_true`. I wonder if these are due to power outages, monitoring station maintenance, or some other issue.  

We can combine `CO_true_in_1_3_str` with a string-cast version of `CO_true_above_1` to understand how often the carbon monoxide concentration is below 1, between 1 and 3, and above 3 units. 

In [25]:
# Casting CO_true_above_1 to string
test_data_sql.df = test_data_sql.df.withColumn("CO_true_above_1_str", F.col("CO_true_above_1").cast("string"))

# Counting observations for combinations of levels for CO_true_in_1_3 and CO_true_above_1
test_data_sql.count_combos(column1 = "CO_true_in_1_3_str", column2 = "CO_true_above_1_str")

,CO_true_in_1_3_str,CO_true_above_1_str,count
0,false,true,1715
1,None,None,1683
2,false,false,1556
3,true,true,4403


The results tell us 1715 observations have a `CO_true` value above 3 units, and 1556 observations have a value below 1. Thus, the overall distribution is very different from what the first 25 observations imply, as four of those observations have a value below 1 compared to only a single observation with a value above 3. 

As was previously mentioned, the input columns should have `string` values. Let's find out what happens when that's not the case. We'll start by passing a numeric column for `column1` and a string column for `column2`. 

In [26]:
# Testing what happens when one column numeric
test_data_sql.count_combos(column1 = "CO_true", column2 = "CO_true_in_1_3_str")

column1 is numeric, should be string


,CO_true_in_1_3_str,count
0,None,1683
1,false,3271
2,true,4403


This is another demonstration that the methods for our custom class are robust to user errors. In this instance, the method returns counts for levels of the string column and also prints a message indicating `column1` is numeric. 

As a final test, we'll pass two numeric columns. 

In [27]:
# Testing what happens when one column numeric
test_data_sql.count_combos(column1 = "CO_true", column2 = "CO_sensor")

Each input column is numeric, but should be string


In this instance, there are no counts to return, so the method simply prints a message indicating that both columns are numeric. 

#### Initializing Class Using `.from_pddf()`

So far, we have manipulated a `SparkDataCheck` object initialized via `.from_csv`. In the final subsection of Part I, let's initialize the class using `.from_pddf` and see if this impacts the dataframe in any way. To use this class method, we'll first need to read the dataset into a `pandas` dataframe. 

In [28]:
# Reading the dataset into a pandas dataframe
pd_df = pd.read_csv("air.csv")

# Creating a SparkDataCheck object containing the air quality data
test_data_sql_2 = SparkDataCheck.from_pddf(spark, pd_df)

Let's now look at the same 25 observations we have seen several times to spot any differences in the way the data are read in. 

In [29]:
# Printing first 25 observations
test_data_sql_2.df.show(25)

+----------+---------+--------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|Unnamed: 0|     Date|    Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|
+----------+---------+--------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|         0|3/10/2004|18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|
|         1|3/10/2004|19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1559|        972|13.3|47.7|0.7255|
|         2|3/10/2004|20:00:00|   2.2|       1402|      88|     9.0|          939|    131|        1140|    114|        1555|       1074|11.9|54.0|0.7502|
|         3|3/10/2004|21:00:00|   2.2|       1376|      80|     9.2|        

Two differences stand out: The first column is now `Unnamed: 0` and `Time` is no longer a time stamp type. Let's figure out exactly how `Time` is being stored. 

In [30]:
# Extracting the type for Time
test_data_sql_2.df.dtypes[2]

('Time', 'string')

It seems that `pandas` assumes `Time` is a string, while `PySpark.sql` identifies it as a time stamp. This is one example of how a `PySpark.sql` dataframe converted from a `pandas` dataframe will not be identical to a `PySpark.sql` dataframe that reads in the same data directly from a file. 

To confirm our methods still work on this new dataframe, let's recreate one of the the original Boolean columns we appended to `test_data_sql`:  `Date_in_set_mar_11`. 

In [31]:
# Adding new boolean column indicating when Date is 3/11/2004
test_data_sql_2.in_set("Date", "3/11/2004", tag = "mar_11").df \
    .select("Date", "Time", "Date_in_set_mar_11").show(25)

+---------+--------+------------------+
|     Date|    Time|Date_in_set_mar_11|
+---------+--------+------------------+
|3/10/2004|18:00:00|             false|
|3/10/2004|19:00:00|             false|
|3/10/2004|20:00:00|             false|
|3/10/2004|21:00:00|             false|
|3/10/2004|22:00:00|             false|
|3/10/2004|23:00:00|             false|
|3/11/2004| 0:00:00|              true|
|3/11/2004| 1:00:00|              true|
|3/11/2004| 2:00:00|              true|
|3/11/2004| 3:00:00|              true|
|3/11/2004| 4:00:00|              true|
|3/11/2004| 5:00:00|              true|
|3/11/2004| 6:00:00|              true|
|3/11/2004| 7:00:00|              true|
|3/11/2004| 8:00:00|              true|
|3/11/2004| 9:00:00|              true|
|3/11/2004|10:00:00|              true|
|3/11/2004|11:00:00|              true|
|3/11/2004|12:00:00|              true|
|3/11/2004|13:00:00|              true|
|3/11/2004|14:00:00|              true|
|3/11/2004|15:00:00|              true|


These results match our expectations and are identical to the original version of `Date_in_set_mar_11`. Thus, we have no reason to believe our methods will behave differently for `SparkDataCheck` objects initialized via `.from_pddf`. 

#### Part I Takeaways

Overall, Part I has highlighted a key tool to overcome the fact that `PySpark.sql` is not as user-friendly as modules like `pandas`: the custom class. Custom classes allow us to reproduce tedious data validation, cleaning, and analysis steps using only a few lines of code, resulting in time savings for future projects that rely on `PySpark.sql`. 

## Part II

When utilizing a `PySpark` kernel, the `PySpark.pandas` and `PySpark.sql` modules offer alternative sets of tools to reap the big-data benefits of Apache Spark. In this section, we will complete the same data manipulations using each module to highlight their relative strengths and weaknesses. 

The dataset to be manipulated contains game-level statistics for NFL players across multiple seasons. As the focus of this exercise is on the `PySpark` modules rather than the data, we won't provide a thorough description of the variables contained in the dataset. Instead, we will describe variables as we use them. 

*Note: The `weekly_nfl_data.csv` file will need to be available in the reader's local environment to run the code chunks in Part II.*

#### Manipulating Data Using `PySpark.pandas`

We will begin by completing all manipulations of the NFL dataset using `PySpark.pandas`, or, in plain(ish) English, `pandas`-on-Spark.

The first steps are to read in the data and print the first few rows to ensure things read in properly. Due to the fact that `PySpark.pandas` effectively crosswalks most `pandas` functionality to `PySpark`, we'll simply use the `read_csv` function and `.head` method to complete these steps. 

In [32]:
# Reading in the NFL data
nfl_df_ps = ps.read_csv("weekly_nfl_data.csv")

# Printing the first five rows
nfl_df_ps.head()

/opt/tljh/user/envs/pySpark3/lib/python3.9/site-packages/pyspark/pandas/utils.py:1037: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `read_csv`, the default index is attached which can cause additional overhead.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


,player_id,player_name,player_display_name,position,position_group,headshot_url,recent_team,season,week,season_type,opponent_team,completions,attempts,passing_yards,passing_tds,interceptions,sacks,sack_yards,sack_fumbles,sack_fumbles_lost,passing_air_yards,passing_yards_after_catch,passing_first_downs,passing_epa,passing_2pt_conversions,pacr,dakota,carries,rushing_yards,rushing_tds,rushing_fumbles,rushing_fumbles_lost,rushing_first_downs,rushing_epa,rushing_2pt_conversions,receptions,targets,receiving_yards,receiving_tds,receiving_fumbles,receiving_fumbles_lost,receiving_air_yards,receiving_yards_after_catch,receiving_first_downs,receiving_epa,receiving_2pt_conversions,racr,target_share,air_yards_share,wopr,special_teams_tds,fantasy_points,fantasy_points_ppr
0,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,MIA,1999,1,REG,DEN,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,16,60.0,1,0.0,0.0,4.0,6.248771,0,1,1,7.0,0,0.0,0.0,0.0,0.0,0.0,0.292378,0,0.0,0.052632,NaN,NaN,0.0,12.7,13.7
1,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,MIA,1999,2,REG,ARI,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,9,33.0,0,0.0,0.0,1.0,-1.434950,0,3,4,18.0,0,0.0,0.0,0.0,0.0,1.0,0.377009,0,0.0,0.117647,NaN,NaN,0.0,5.1,8.1
2,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,MIA,1999,4,REG,BUF,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,3,2.0,0,0.0,0.0,0.0,-1.539952,0,0,1,0.0,0,0.0,0.0,0.0,0.0,0.0,-0.699578,0,NaN,0.023810,NaN,NaN,0.0,0.2,0.2
3,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,CLE,1999,7,REG,LA,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,6,27.0,0,0.0,0.0,0.0,0.216051,0,2,2,8.0,0,0.0,0.0,0.0,0.0,0.0,-0.228454,0,0.0,0.050000,NaN,NaN,0.0,3.5,5.5
4,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,CLE,1999,8,REG,NO,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,13,39.0,0,0.0,0.0,2.0,-2.972259,0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,NaN,0,NaN,NaN,NaN,NaN,0.0,3.9,3.9


Looking at the first five rows, it seems the data were read in with no issue. This dataset contains a tremendous amount of information on individual players' games! We should not overlook the fact that the `pandas`-style printing makes this very wide dataset manageable to explore.

The reader may note the warning printed above. It is not concerning and simply tells the user to specify an index column so that computing resources aren't wasted creating a new index.

We can use the `.column` attribute to easily access the full set of column names. Note that we'll use the `.to_list()` method to explicitly extract to column names from the `pandas` object they are stored in. 

In [33]:
# Printing column names
nfl_df_ps.columns.to_list()

['player_id',
 'player_name',
 'player_display_name',
 'position',
 'position_group',
 'headshot_url',
 'recent_team',
 'season',
 'week',
 'season_type',
 'opponent_team',
 'completions',
 'attempts',
 'passing_yards',
 'passing_tds',
 'interceptions',
 'sacks',
 'sack_yards',
 'sack_fumbles',
 'sack_fumbles_lost',
 'passing_air_yards',
 'passing_yards_after_catch',
 'passing_first_downs',
 'passing_epa',
 'passing_2pt_conversions',
 'pacr',
 'dakota',
 'carries',
 'rushing_yards',
 'rushing_tds',
 'rushing_fumbles',
 'rushing_fumbles_lost',
 'rushing_first_downs',
 'rushing_epa',
 'rushing_2pt_conversions',
 'receptions',
 'targets',
 'receiving_yards',
 'receiving_tds',
 'receiving_fumbles',
 'receiving_fumbles_lost',
 'receiving_air_yards',
 'receiving_yards_after_catch',
 'receiving_first_downs',
 'receiving_epa',
 'receiving_2pt_conversions',
 'racr',
 'target_share',
 'air_yards_share',
 'wopr',
 'special_teams_tds',
 'fantasy_points',
 'fantasy_points_ppr']

The list of column names is useful when we want to loop over each column and apply some operation. 

We're now ready to manipulate this dataframe. Let's subset to only quarterbacks and their key stats in the regular season (2005 through 2023 only) using `.loc[]`, aggregate the statistics to the season level sums and means using `.groupby()` in combination with `.agg()`. We'll print the first 10 rows to check the outcome. 

In [34]:
# aggregating regular season QB stats to season level for 2005 to 2023 seasons
nfl_df_ps_qb = nfl_df_ps \
    .loc[(nfl_df_ps.season_type == "REG") & (nfl_df_ps.position == "QB") & \
         (nfl_df_ps.season.isin(range(2005, 2024))), 
         ["player_display_name", "season", "week", "completions",
    "attempts", "passing_yards", "passing_tds", "interceptions"]] \
    .groupby(["player_display_name", "season"]) \
    [["completions", "attempts", "passing_yards", 
     "passing_tds", "interceptions"]] \
    .agg(["sum", "mean"])

# Printing first 10 rows
nfl_df_ps_qb.head(10)

completions            attempts            passing_yards             passing_tds           interceptions          
                                   sum       mean      sum       mean           sum        mean         sum      mean           sum      mean
player_display_name season                                                                                                                   
Jake Delhomme       2006           263  20.230769      431  33.153846        2805.0  215.769231          17  1.307692          11.0  0.846154
Jake Plummer        2005           277  17.312500      456  28.500000        3366.0  210.375000          18  1.125000           7.0  0.437500
Matt Schaub         2006            18   3.600000       27   5.400000         208.0   41.600000           1  0.200000           2.0  0.400000
Vince Young         2006           184  12.266667      356  23.733333        2199.0  146.600000          12  0.800000          13.0  0.866667
Kerry Collins       2007            50   8.333333       82  13.666667         531.0   88.500000           0  0.000000           0.0  0.000000
Todd Collins        2005             0   0.000000        0   0.000000           0.0    0.000000           0  0.000000           0.0  0.000000
Charlie Batch       2006            30   4.285714       52   7.428571         477.0   68.142857           5  0.714286           0.0  0.000000
Eli Manning         2006           301  18.812500      522  32.625000        3244.0  202.750000          25  1.562500          18.0  1.125000
A.J. Feeley         2006            26  13.000000       38  19.000000         342.0  171.000000           3  1.500000           0.0  0.000000
Mark Brunell        2006           162  16.200000      261  26.100000        1789.0  178.900000           8  0.800000           4.0  0.400000

This gives us the dataset we want, but the hierarchical structure of the columns is rather inconvenient, as is the multi-level index that uses both `player_display_name` and `season`. This is a clear weakness of standard `pandas` aggregation approaches. 

Let's try this again. This time, we will use a dictionary comprehension to create a dictionary of aggregated statistic names and their corresponding input column plus aggregation metric (`sum` or `mean`). The dictionary comprehension utilizes f-strings to create entries such as `"completions_sum": ("completions", "sum")`, which is exactly what we need to generate clean season level statistics columns with no hierarchy. We'll then unpack this dictionary within the `.agg()` method to generate the same season-level statistics as above. Finally, we will use the `.reset_index()` method to drop the multi-level index. 

In [35]:
# capturing columns to be aggregated and functions
columns = ["completions", "attempts", "passing_yards", 
     "passing_tds", "interceptions"]
agg_funcs = ["sum", "mean"]

# Using dictionary comprehension to capture text for .agg
# using an f string
stats_dict = {f"{c}_{fun}": (c, fun)
                  for c in columns
                  for fun in agg_funcs}

# aggregating regular season QB stats to season level
# by unpacking the stats_dict
# resetting index at the end to remove multi-level index
nfl_df_ps_qb = nfl_df_ps \
    .loc[(nfl_df_ps.season_type == "REG") & (nfl_df_ps.position == "QB") & 
         (nfl_df_ps.season.isin(range(2005, 2024))), 
         ["player_display_name", "season", "week", "completions",
    "attempts", "passing_yards", "passing_tds", "interceptions"]] \
    .groupby(["player_display_name", "season"]) \
    .agg(**stats_dict).reset_index()

# Printing first 10 rows
nfl_df_ps_qb.head(10)

,player_display_name,season,completions_sum,completions_mean,attempts_sum,attempts_mean,passing_yards_sum,passing_yards_mean,passing_tds_sum,passing_tds_mean,interceptions_sum,interceptions_mean
0,Jake Delhomme,2006,263,20.230769,431,33.153846,2805.0,215.769231,17,1.307692,11.0,0.846154
1,Jake Plummer,2005,277,17.312500,456,28.500000,3366.0,210.375000,18,1.125000,7.0,0.437500
2,Matt Schaub,2006,18,3.600000,27,5.400000,208.0,41.600000,1,0.200000,2.0,0.400000
3,Vince Young,2006,184,12.266667,356,23.733333,2199.0,146.600000,12,0.800000,13.0,0.866667
4,Kerry Collins,2007,50,8.333333,82,13.666667,531.0,88.500000,0,0.000000,0.0,0.000000
5,Todd Collins,2005,0,0.000000,0,0.000000,0.0,0.000000,0,0.000000,0.0,0.000000
6,Charlie Batch,2006,30,4.285714,52,7.428571,477.0,68.142857,5,0.714286,0.0,0.000000
7,Eli Manning,2006,301,18.812500,522,32.625000,3244.0,202.750000,25,1.562500,18.0,1.125000
8,A.J. Feeley,2006,26,13.000000,38,19.000000,342.0,171.000000,3,1.500000,0.0,0.000000
9,Mark Brunell,2006,162,16.200000,261,26.100000,1789.0,178.900000,8,0.800000,4.0,0.400000


This admittedly does not look as nice as our original result and took more work to create, but it will be much easier to work with. 

Let's document exactly what we have: total and game-level average completions, attempts, passing yards, passing touchdowns, and interceptions. 

We should add a few additional metrics: 
 - `completion_percentage`: the season-level ratio of completions to attempts (multiplied by 100)
 - `td_int_ratio`: the season-level ratio of passing touchdowns to interceptions
 
 These additions would have been very difficult if we had retained the original structure for `nfl_df_ps_qb`. Now, we can simply use assignment. 
 
 We'll again print out the first 10 rows to check the results. 

In [36]:
# Adding season completion percentage to dataframe
nfl_df_ps_qb["completion_percentage"] = 100*nfl_df_ps_qb.completions_sum/nfl_df_ps_qb.attempts_sum

# Adding season td-int ratio to dataframe
nfl_df_ps_qb["td_int_ratio"] = nfl_df_ps_qb.passing_tds_sum/nfl_df_ps_qb.interceptions_sum

# Printing first 10 rows
nfl_df_ps_qb.head(10)

,player_display_name,season,completions_sum,completions_mean,attempts_sum,attempts_mean,passing_yards_sum,passing_yards_mean,passing_tds_sum,passing_tds_mean,interceptions_sum,interceptions_mean,completion_percentage,td_int_ratio
0,Jake Delhomme,2006,263,20.230769,431,33.153846,2805.0,215.769231,17,1.307692,11.0,0.846154,61.020882,1.545455
1,Jake Plummer,2005,277,17.312500,456,28.500000,3366.0,210.375000,18,1.125000,7.0,0.437500,60.745614,2.571429
2,Matt Schaub,2006,18,3.600000,27,5.400000,208.0,41.600000,1,0.200000,2.0,0.400000,66.666667,0.500000
3,Vince Young,2006,184,12.266667,356,23.733333,2199.0,146.600000,12,0.800000,13.0,0.866667,51.685393,0.923077
4,Kerry Collins,2007,50,8.333333,82,13.666667,531.0,88.500000,0,0.000000,0.0,0.000000,60.975610,NaN
5,Todd Collins,2005,0,0.000000,0,0.000000,0.0,0.000000,0,0.000000,0.0,0.000000,NaN,NaN
6,Charlie Batch,2006,30,4.285714,52,7.428571,477.0,68.142857,5,0.714286,0.0,0.000000,57.692308,inf
7,Eli Manning,2006,301,18.812500,522,32.625000,3244.0,202.750000,25,1.562500,18.0,1.125000,57.662835,1.388889
8,A.J. Feeley,2006,26,13.000000,38,19.000000,342.0,171.000000,3,1.500000,0.0,0.000000,68.421053,inf
9,Mark Brunell,2006,162,16.200000,261,26.100000,1789.0,178.900000,8,0.800000,4.0,0.400000,62.068966,2.000000


Vince Young may have been an incredible college quarterback, but he threw more interceptions than touchdowns during his rookie season in the NFL.... 

If we want to subset this dataframe, we can easily do so using `.loc[]`. Let's filter to only quarterbacks with more than 50 pass attempts in the given season so that we can hone in on players who likely started one or more games. 

In [37]:
# Filtering to only quarterback-season combos where attempts at least 50
nfl_df_ps_qb = nfl_df_ps_qb.loc[nfl_df_ps_qb.attempts_sum >= 50]

We can now sort (descending) by `completion_percentage` to see who posted the best percentages across the 2005 through 2023 seasons. We can sort by simply using the `.sort_values()` method and again use `.head()` to print only the top 40 player-season combinations. I wonder how many of these seasons will belong to the hyper-accurate Drew Brees. 

In [38]:
# Sorting by completion_percentage descending
nfl_df_ps_qb.sort_values("completion_percentage", ascending = False).head(40)

,player_display_name,season,completions_sum,completions_mean,attempts_sum,attempts_mean,passing_yards_sum,passing_yards_mean,passing_tds_sum,passing_tds_mean,interceptions_sum,interceptions_mean,completion_percentage,td_int_ratio
1409,C.J. Beathard,2023,40,6.666667,53,8.833333,349.0,58.166667,1,0.166667,0.0,0.000000,75.471698,inf
1248,Colt McCoy,2021,74,10.571429,99,14.142857,740.0,105.714286,3,0.428571,1.0,0.142857,74.747475,3.000000
900,Matt Schaub,2019,50,10.000000,67,13.400000,580.0,116.000000,3,0.600000,1.0,0.200000,74.626866,3.000000
953,Drew Brees,2018,364,24.266667,489,32.600000,3992.0,266.133333,32,2.133333,5.0,0.333333,74.437628,6.400000
1057,Drew Brees,2019,281,25.545455,378,34.363636,2979.0,270.818182,27,2.454545,4.0,0.363636,74.338624,6.750000
1338,Mason Rudolph,2023,55,13.750000,74,18.500000,719.0,179.750000,3,0.750000,0.0,0.000000,74.324324,inf
1133,Taysom Hill,2020,88,5.500000,121,7.562500,928.0,58.000000,4,0.250000,2.0,0.125000,72.727273,2.000000
1007,Nick Foles,2018,141,28.200000,195,39.000000,1413.0,282.600000,7,1.400000,4.0,0.800000,72.307692,1.750000
917,Drew Brees,2017,386,24.125000,536,33.500000,4334.0,270.875000,23,1.437500,8.0,0.500000,72.014925,2.875000
851,Sam Bradford,2016,395,26.333333,552,36.800000,3877.0,258.466667,20,1.333333,5.0,0.333333,71.557971,4.000000


That's a lot of `Drew Brees` seasons.... 

At first glance, it seems an attempts threshold of 50 may still be too low, but we won't worry about that; what matters is that filtering and sorting is very easy within the `pandas` framework. 

As a final step, let's sort (descending) by `td_int_ratio` to see which quarterbacks managed to post prolific scoring numbers without flying too close to the "sun", a.k.a. Darrelle Revis. 

In [39]:
# Sorting by td_int_ratio descending
nfl_df_ps_qb.sort_values("td_int_ratio", ascending = False).head(40)

,player_display_name,season,completions_sum,completions_mean,attempts_sum,attempts_mean,passing_yards_sum,passing_yards_mean,passing_tds_sum,passing_tds_mean,interceptions_sum,interceptions_mean,completion_percentage,td_int_ratio
6,Charlie Batch,2006,30,4.285714,52,7.428571,477.0,68.142857,5,0.714286,0.0,0.000000,57.692308,inf
26,Matt Schaub,2005,33,4.714286,64,9.142857,495.0,70.714286,4,0.571429,0.0,0.000000,51.562500,inf
73,Todd Collins,2007,67,16.750000,105,26.250000,888.0,222.000000,5,1.250000,0.0,0.000000,63.809524,inf
455,Troy Smith,2007,40,10.000000,76,19.000000,452.0,113.000000,2,0.500000,0.0,0.000000,52.631579,inf
520,Jake Locker,2011,34,6.800000,66,13.200000,542.0,108.400000,4,0.800000,0.0,0.000000,51.515152,inf
775,Brian Hoyer,2016,134,22.333333,200,33.333333,1445.0,240.833333,6,1.000000,0.0,0.000000,67.000000,inf
778,Nick Foles,2016,36,18.000000,55,27.500000,410.0,205.000000,3,1.500000,0.0,0.000000,65.454545,inf
812,Derek Anderson,2014,65,13.000000,97,19.400000,701.0,140.200000,5,1.000000,0.0,0.000000,67.010309,inf
940,Jimmy Garoppolo,2016,43,7.166667,63,10.500000,502.0,83.666667,4,0.666667,0.0,0.000000,68.253968,inf
984,Matt Moore,2019,59,9.833333,91,15.166667,659.0,109.833333,4,0.666667,0.0,0.000000,64.835165,inf


Again, the results are dominated by quarterbacks who were not full-time starters during the corresponding season; the top 15 ratios are all infinite. 

Once we scroll down to players with a few hundred attempts, we still see some very impressive statistics: in 2020, Aaron Rodgers threw 48 touchdowns compared to only 5 interceptions. 

This completes our data manipulations using `pandas`-on-Spark. Overall, no steps were particularly difficult, but the multi-level indexes and hierarchical columns produced by `.agg()` are weaknesses that need to be overcome to streamline aggregation for large datasets. 

#### Manipulating Data Using `PySpark.sql`

Let's recreate the same steps from the previous subsection using the tools available in the `PySpark.sql` module. 

We'll start by reading in the data from the csv file and extracting the first five rows. To read in the data, we'll use the generic `spark.read.load()` function with `format = "csv"` and `sep = ","` to indicate the file we are reading in is comma-delimited. We'll also tell the function we want it to infer the schema based on the data in each column (`inferSchema = "true"`) and indicate the file contains column names (`header = "true"`). The read step is less convenient than `ps.read_csv()`, but it is not excessively tedious. To extract the first five observations of the new `PySpark.sql` dataframe, we can use the `.show()` method; other dataframe-printing options exist, but this option provides the cleanest results. 

In [40]:
# Reading in data
nfl_df_sql = spark.read.load("weekly_nfl_data.csv",
                     format="csv", 
                     sep=",", 
                     inferSchema="true", 
                     header="true")

# Extracting first five rows
nfl_df_sql.show(5)

+----------+-----------+--------------------+--------+--------------+------------+-----------+------+----+-----------+-------------+-----------+--------+-------------+-----------+-------------+-----+----------+------------+-----------------+-----------------+-------------------------+-------------------+-----------+-----------------------+----+------+-------+-------------+-----------+---------------+--------------------+-------------------+-----------+-----------------------+----------+-------+---------------+-------------+-----------------+----------------------+-------------------+---------------------------+---------------------+-------------+-------------------------+----+------------+---------------+----+-----------------+--------------+------------------+
| player_id|player_name| player_display_name|position|position_group|headshot_url|recent_team|season|week|season_type|opponent_team|completions|attempts|passing_yards|passing_tds|interceptions|sacks|sack_yards|sack_fumbles|sack_

Upon printing the first five rows, we immediately see a weakness of `PySpark.sql`: dataframe printing is much less clean. Of course, we could easily convert our SQL-style dataframe to a `PySpark.pandas` dataframe to obtain cleaner printing. However, the goal of this exercise is to compare the strengths and weaknesses of the two modules. 

To confirm the data were read in correctly, we can print only the first 15 columns for the first five observations. To do so, we'll slice the dataframe's `.columns` attribute and pass the subset of columns to the `.select()` method that mimicks a SQL `SELECT` statement. 

In [41]:
# Extracting first five rows
nfl_df_sql.select(nfl_df_sql.columns[0:15]).show(5)

+----------+-----------+--------------------+--------+--------------+------------+-----------+------+----+-----------+-------------+-----------+--------+-------------+-----------+
| player_id|player_name| player_display_name|position|position_group|headshot_url|recent_team|season|week|season_type|opponent_team|completions|attempts|passing_yards|passing_tds|
+----------+-----------+--------------------+--------+--------------+------------+-----------+------+----+-----------+-------------+-----------+--------+-------------+-----------+
|00-0000003|       NULL|Abdul-Karim al-Ja...|      RB|            RB|        NULL|        MIA|  1999|   1|        REG|          DEN|          0|       0|          0.0|          0|
|00-0000003|       NULL|Abdul-Karim al-Ja...|      RB|            RB|        NULL|        MIA|  1999|   2|        REG|          ARI|          0|       0|          0.0|          0|
|00-0000003|       NULL|Abdul-Karim al-Ja...|      RB|            RB|        NULL|        MIA|  1999

Based on the first 15 columns, it seems the data were read in correctly. Looking at `player_name`, we see another difference between `PySpark.pandas` and `PySpark.sql` dataframes: In keeping with SQL conventions, missing values in the `PySpark.sql` dataframe are denoted `NULL` rather than the Pythonic `None` or `NaN` values. 

As the reader might expect based on the previous code chunk, we can access the dataframe's columns using the `.columns` attribute. 

In [42]:
# Extract column names
nfl_df_sql.columns

['player_id',
 'player_name',
 'player_display_name',
 'position',
 'position_group',
 'headshot_url',
 'recent_team',
 'season',
 'week',
 'season_type',
 'opponent_team',
 'completions',
 'attempts',
 'passing_yards',
 'passing_tds',
 'interceptions',
 'sacks',
 'sack_yards',
 'sack_fumbles',
 'sack_fumbles_lost',
 'passing_air_yards',
 'passing_yards_after_catch',
 'passing_first_downs',
 'passing_epa',
 'passing_2pt_conversions',
 'pacr',
 'dakota',
 'carries',
 'rushing_yards',
 'rushing_tds',
 'rushing_fumbles',
 'rushing_fumbles_lost',
 'rushing_first_downs',
 'rushing_epa',
 'rushing_2pt_conversions',
 'receptions',
 'targets',
 'receiving_yards',
 'receiving_tds',
 'receiving_fumbles',
 'receiving_fumbles_lost',
 'receiving_air_yards',
 'receiving_yards_after_catch',
 'receiving_first_downs',
 'receiving_epa',
 'receiving_2pt_conversions',
 'racr',
 'target_share',
 'air_yards_share',
 'wopr',
 'special_teams_tds',
 'fantasy_points',
 'fantasy_points_ppr']

I can't imagine an easier process to access columns. The list confirms that all columns were correctly identified when we read in the data, a something that was hard to check by printing the dataframe. 

We are now ready to manipulate this dataframe. This is where `PySpark.sql` will shine, as it brings the data-manipulation advantages of SQL. In fact, we can run a SQL query directly by first creating a temporary view of `nfl_df_sql` using the `createTempView()` method, then using the `spark.sql()` function to apply a SQL query to the temporary view. We will use this process to generate the season-level statistics for quarterbacks that we created in the previous subsection.

The SQL query itself will do the following:
- Use a `SELECT` statement to indicate we want to aggregate key quarterback statistics by both summing and averaging across weeks
    - The `SELECT` statement will also be used to define `completion_percentage` and `td_int_ratio`, saving a step in comparison to the `pandas`-on-Spark process
- Use a `WHERE` statement to indicate we only want regular season games for quarterbacks from the 2005 through 2023 seasons
- Use a `GROUP BY` statement to indicate we want to aggregate data to the player-season level

After we create the new dataframe using the SQL query, we'll extract the first 10 rows to check the results. 

In [43]:
# Creating a temporary view of nfl_df_sql to run SQL query on
nfl_df_sql.createTempView("df")

# Running SQL query on temporary view
# Aggregating key QB statistics to the player-season level as sums and averages
# Only keeping regular season statistics from 2005 to 2023 for quarterbacks
# Also calculating season-level completion_percentage and td_int_ratio
nfl_df_sql_qb = spark.sql("""
    SELECT player_display_name, season, 
        sum(completions) as completions_sum,
        avg(completions) as completions_mean,
        sum(attempts) as attempts_sum,
        avg(attempts) as attempts_mean,
        sum(passing_yards) as passing_yards_sum,
        avg(passing_yards) as passing_yards_mean,
        sum(passing_tds) as passing_tds_sum,
        avg(passing_tds) as passing_tds_mean,
        sum(interceptions) as interceptions_sum,
        avg(interceptions) as interceptions_mean,
        100*sum(completions)/sum(attempts) as completion_percentage,
        sum(passing_tds)/sum(interceptions) as td_int_ratio
    FROM df
    WHERE season_type = "REG" AND position = "QB"
        AND season >= 2005 AND season <= 2023
    GROUP BY player_display_name, season
""")

# Extracting the first 10 rows of the new dataframe
nfl_df_sql_qb.show(10)


+-------------------+------+---------------+------------------+------------+------------------+-----------------+------------------+---------------+------------------+-----------------+------------------+---------------------+------------------+
|player_display_name|season|completions_sum|  completions_mean|attempts_sum|     attempts_mean|passing_yards_sum|passing_yards_mean|passing_tds_sum|  passing_tds_mean|interceptions_sum|interceptions_mean|completion_percentage|      td_int_ratio|
+-------------------+------+---------------+------------------+------------+------------------+-----------------+------------------+---------------+------------------+-----------------+------------------+---------------------+------------------+
|      Jake Delhomme|  2006|            263| 20.23076923076923|         431| 33.15384615384615|           2805.0|215.76923076923077|             17|1.3076923076923077|             11.0|0.8461538461538461|    61.02088167053364|1.5454545454545454|
|       Jake Plu

Ah yes, the messy printing strikes again.... However, the overrun of each row is not as severe as it was when we attempted to extract the first few rows of `nfl_df_sql`, allowing us to read the table even if it is difficult to do so; for some wide screens the table may be relatively clean with no overrun.  

In general, the results are the same as those produced by our `PySpark.pandas` code, but there is one notable difference when the quarterback has no interceptions in a season:
- `PySpark.pandas`:
    - When the quarterback also has no touchdowns, `td_int_ratio` is `NaN`
    - When the quarterback has at least one touchdown, `td_int_ratio` is `Inf`
- `PySpark.sql`: `td_int_ratio` is `NULL` regardless of the number of touchdowns

Overall, for those comfortable with SQL queries, `PySpark.sql` provides useful alternatives to `pandas`-style data manipulation.

Let's now subset our new `nfl_df_sql_qb` dataframe to only player-season combinations with more than 50 attempts using the `.where()` method that mimicks the behavior of a `WHERE` statement. 

In [44]:
# Subsetting to player-season combos with at least 50 attempts
nfl_df_sql_qb = nfl_df_sql_qb.where(F.col("attempts_sum") >= 50)

This step was no more or less difficult than the same step applied to the `PySpark.pandas` dataframe. 

We are now ready to sort the dataframe by `completion_percentage` to return the top 40 season-long performances based on this metric. We'll use the `.orderBy()` method, which mimicks the SQL `ORDER BY` statement. 

In [45]:
# Sorting by completion_percentage descending
nfl_df_sql_qb.orderBy(F.col("completion_percentage").desc()).show(40)

+-------------------+------+---------------+------------------+------------+------------------+-----------------+------------------+---------------+-------------------+-----------------+-------------------+---------------------+------------------+
|player_display_name|season|completions_sum|  completions_mean|attempts_sum|     attempts_mean|passing_yards_sum|passing_yards_mean|passing_tds_sum|   passing_tds_mean|interceptions_sum| interceptions_mean|completion_percentage|      td_int_ratio|
+-------------------+------+---------------+------------------+------------+------------------+-----------------+------------------+---------------+-------------------+-----------------+-------------------+---------------------+------------------+
|      C.J. Beathard|  2023|             40| 6.666666666666667|          53| 8.833333333333334|            349.0|58.166666666666664|              1|0.16666666666666666|              0.0|                0.0|    75.47169811320755|              NULL|
|       

Again, the results are generally the same as those generated via the `PySpark.pandas` module and are equally simply to produce, even if the table is not as visually appealing. 

Our final coding step is to sort the dataframe by `td_int_ratio` and print the top 40 seasons by a quarterback with at least 50 attempts. We will also complete this step using `.groupBy()`. 

In [47]:
# Sorting by td_int_ratio descending
nfl_df_sql_qb.orderBy(F.col("td_int_ratio").desc()).show(40)

+-------------------+------+---------------+------------------+------------+------------------+-----------------+------------------+---------------+-------------------+-----------------+-------------------+---------------------+-------------------+
|player_display_name|season|completions_sum|  completions_mean|attempts_sum|     attempts_mean|passing_yards_sum|passing_yards_mean|passing_tds_sum|   passing_tds_mean|interceptions_sum| interceptions_mean|completion_percentage|       td_int_ratio|
+-------------------+------+---------------+------------------+------------+------------------+-----------------+------------------+---------------+-------------------+-----------------+-------------------+---------------------+-------------------+
|      Kerry Collins|  2007|             50| 8.333333333333334|          82|13.666666666666666|            531.0|              88.5|              0|                0.0|              0.0|                0.0|    60.97560975609756|               NULL|
|   

In this case, there are substantial differences in what we print: Because `PySpark.pandas` assigns some quarterbacks' seasons a touchdown-to-interception ratio of infinity, those seasons are ranked at the top when we sort by the metric. However, `PySpark.sql` considers the same ratios to be `NULL`, so the seasons are placed at the bottom of the dataframe. 

#### Takeaways from Comparison Exercise

Overall, both `PySpark.pandas` and `PySpark.sql` have strengths and weaknesses. While many operations are similarly complex across the two modules, `PySpark.sql` offers the advantage of accepting explicit SQL code to simplify complex manipulations. Contrarily, data printing methods are superior in `PySpark.pandas`. Overall, the most effective approach will often be to combine the two modules when data tasks are complex and the data scientist is knowledgeable of SQL. 

Arguably more important than selecting the most efficient module for a given task is understanding how the selected module handles edge cases in the data. In exploring the touchdown-to-interception ratio, we saw that `PySpark.pandas` and `PySpark.sql` handle division by 0 differently. This is only one example of the nuanced behavior of each module, highlighting the need for the data scientist to fully understand each module to use it appropriately and accurately. 

## Conclusion

Throughout this notebook, we have explored the capabilities of the `PySpark` API by verifying and manipulating data using the `PySpark.pandas` and `PySpark.sql` modules. In Part I, we highlighted the ability to use custom classes and corresponding methods to streamline verification steps that are often tedious when using the `PySpark.sql` module alone. In Part II, we compared the data manipulation capabilities of the two modules, demonstrating the relative strengths and weaknesses of each as well as their nuanced differences in behavior. 

The main takeaway from this notebook is that `PySpark.pandas` and `PySpark.sql` are two valuable tools to unlocking the big data capabilities of Spark via Python. These tools are even more powerful when used together, but only when the user fully understands their behavior at edge cases because such cases are often present in big data. 